# MetaQuest VR SO100 Robot Control
## MetaQuest VR 헤드셋을 사용하여 SO100 로봇 제어

이 노트북은 MetaQuest VR 헤드셋을 사용하여 SO100 로봇을 제어하는 시스템입니다.

**3단계로 나눠서 실행:**
1. 🤖 **로봇 연결** - SO100 로봇과 통신 시작
2. 📱 **MetaQuest 접속** - VR 헤드셋 연결 준비  
3. 🎮 **제어 시작** - VR로 팔 조종 시작

In [1]:
# libraries and classes for robot control and VR integration
import asyncio
import logging
import math
import os
import sys
import threading
import time
import traceback
import socket
from typing import Optional, Dict, Any

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Store global variables for use across cells
globals_dict = {
    'robot': None,
    'vr_monitor': None,
    'controller': None,
    'event_loop_thread': None,
    'event_loop': None,
}

# Helper class for managing asyncio event loop in a separate thread
class AsyncEventLoopThread(threading.Thread):
    """Thread that runs an asyncio event loop"""
    def __init__(self):
        super().__init__(daemon=True)
        self.loop = None
        self.ready = threading.Event()
        
    def run(self):
        """Run the event loop in this thread"""
        self.loop = asyncio.new_event_loop()
        asyncio.set_event_loop(self.loop)
        self.ready.set()
        self.loop.run_forever()
    
    def stop(self):
        """Stop the event loop"""
        if self.loop and self.loop.is_running():
            self.loop.call_soon_threadsafe(self.loop.stop)
    
    def run_coroutine(self, coro):
        """Run a coroutine in the event loop"""
        if self.loop and self.loop.is_running():
            return asyncio.run_coroutine_threadsafe(coro, self.loop)
        return None

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [2]:
# XLeVR Configuration
XLEVR_PATH = "/home/choyunsang/XLeRobot/XLeVR"

def get_local_ip():
    """Get local machine IP address"""
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:
            s.connect(("8.8.8.8", 80))
            return s.getsockname()[0]
    except Exception:
        try:
            return socket.gethostbyname(socket.gethostname())
        except Exception:
            return "localhost"

def setup_xlevr_environment():
    """Setup XLeVR environment"""
    if XLEVR_PATH not in sys.path:
        sys.path.insert(0, XLEVR_PATH)
    os.chdir(XLEVR_PATH)
    os.environ['PYTHONPATH'] = f"{XLEVR_PATH}:{os.environ.get('PYTHONPATH', '')}"
    logger.info(f"✅ XLeVR environment configured: {XLEVR_PATH}")

def import_xlevr_modules():
    """Import XLeVR modules"""
    try:
        from xlevr.config import XLeVRConfig
        from xlevr.inputs.vr_ws_server import VRWebSocketServer
        from xlevr.inputs.base import ControlGoal, ControlMode
        logger.info("✅ XLeVR modules imported")
        return XLeVRConfig, VRWebSocketServer, ControlGoal, ControlMode
    except ImportError as e:
        logger.error(f"❌ Failed to import XLeVR modules: {e}")
        return None, None, None, None

# Import SSL and HTTP modules for HTTPS server
import ssl
import http.server

class SimpleAPIHandler(http.server.BaseHTTPRequestHandler):
    """HTTP request handler for serving web UI"""
    
    def end_headers(self):
        """Add CORS headers to all responses."""
        self.send_header('Access-Control-Allow-Origin', '*')
        self.send_header('Access-Control-Allow-Methods', 'GET, POST, OPTIONS')
        self.send_header('Access-Control-Allow-Headers', 'Content-Type')
        try:
            super().end_headers()
        except (BrokenPipeError, ConnectionResetError, ConnectionAbortedError, ssl.SSLError):
            pass
    
    def do_OPTIONS(self):
        """Handle preflight CORS requests."""
        self.send_response(200)
        self.end_headers()
    
    def log_message(self, format, *args):
        """Override to reduce HTTP request logging noise."""
        pass  # Disable default HTTP logging
    
    def do_GET(self):
        """Handle GET requests."""
        if self.path == '/' or self.path == '/index.html':
            self.serve_file('web-ui/index.html', 'text/html')
        elif self.path.endswith('.css'):
            self.serve_file(f'web-ui{self.path}', 'text/css')
        elif self.path.endswith('.js'):
            self.serve_file(f'web-ui{self.path}', 'application/javascript')
        elif self.path.endswith(('.jpg', '.jpeg', '.png', '.gif')):
            content_type = 'image/jpeg' if self.path.endswith(('.jpg', '.jpeg')) else 'image/png' if self.path.endswith('.png') else 'image/gif'
            self.serve_file(f'web-ui{self.path}', content_type)
        else:
            self.send_error(404, "Not found")
    
    def serve_file(self, filename, content_type):
        """Serve a file with the given content type."""
        try:
            file_path = os.path.join(XLEVR_PATH, filename)
            
            if os.path.exists(file_path):
                with open(file_path, 'rb') as f:
                    content = f.read()
                
                self.send_response(200)
                self.send_header('Content-Type', content_type)
                self.end_headers()
                self.wfile.write(content)
            else:
                self.send_error(404, f"File not found: {filename}")
        except Exception as e:
            logger.error(f"Error serving file {filename}: {e}")
            self.send_error(500, "Internal server error")

class SimpleHTTPSServer:
    """HTTPS server for providing web interface"""
    
    def __init__(self, host, port):
        self.host = host
        self.port = port
        self.httpd = None
        self.server_thread = None
    
    async def start(self):
        """Start the HTTPS server."""
        try:
            self.httpd = http.server.HTTPServer((self.host, self.port), SimpleAPIHandler)
            
            # Setup SSL
            context = ssl.SSLContext(ssl.PROTOCOL_TLS_SERVER)
            context.load_cert_chain(
                os.path.join(XLEVR_PATH, 'cert.pem'),
                os.path.join(XLEVR_PATH, 'key.pem')
            )
            self.httpd.socket = context.wrap_socket(self.httpd.socket, server_side=True)
            
            # Start server in a separate thread
            self.server_thread = threading.Thread(target=self.httpd.serve_forever, daemon=True)
            self.server_thread.start()
            logger.info(f"✅ HTTPS server started on {self.host}:{self.port}")
            
        except Exception as e:
            logger.error(f"❌ Failed to start HTTPS server: {e}")
            raise
    
    async def stop(self):
        """Stop the HTTPS server."""
        if self.httpd:
            self.httpd.shutdown()
            if self.server_thread:
                self.server_thread.join(timeout=5)
            logger.info("✅ HTTPS server stopped")

# Setup environment on first import
setup_xlevr_environment()
print("✅ XLeVR environment configured")

2026-05-20 19:21:05,060 - __main__ - INFO - ✅ XLeVR environment configured: /home/choyunsang/XLeRobot/XLeVR


✅ XLeVR environment configured


In [3]:
# VR Monitor Class - for SO100
class VRControlMonitor:
    """VR controller monitor for MetaQuest - SO100 single arm"""
    
    def __init__(self):
        self.config = None
        self.vr_server = None
        self.https_server = None
        self.is_running = False
        self.right_goal = None
        self._goal_lock = threading.Lock()
        self.command_queue = None
        self.monitoring_task = None
        self.last_update_time = time.time()
        
    def initialize(self):
        """Initialize VR monitor - SO100 single arm"""
        logger.info("🔧 Initializing VR Monitor (SO100)...")
        
        XLeVRConfig, VRWebSocketServer, _, _ = import_xlevr_modules()
        if XLeVRConfig is None:
            logger.error("❌ Failed to import XLeVR modules")
            return False
        
        self.config = XLeVRConfig()
        self.config.enable_vr = True
        self.config.enable_keyboard = False
        self.command_queue = asyncio.Queue()
        
        try:
            # Create VR WebSocket server
            self.vr_server = VRWebSocketServer(
                command_queue=self.command_queue,
                config=self.config,
                print_only=False
            )
            logger.info("✅ VR WebSocket server created")
            
            # Create HTTPS web UI server
            host_ip = self.config.host_ip if self.config.host_ip != "0.0.0.0" else "0.0.0.0"
            https_port = self.config.https_port if hasattr(self.config, 'https_port') else 8443
            
            self.https_server = SimpleHTTPSServer(host_ip, https_port)
            logger.info("✅ HTTPS web server created")
            
            # Display connection info
            host_ip_display = get_local_ip() if self.config.host_ip == "0.0.0.0" else self.config.host_ip
            websocket_port = self.config.websocket_port if hasattr(self.config, 'websocket_port') else 8442
            
            print("\n" + "="*70)
            print("📱 MetaQuest VR Headset - COPY THIS URL:")
            print(f"\n   https://{host_ip_display}:{https_port}\n")
            print(f"Connection Details:")
            print(f"  • HTTPS Web UI: {host_ip_display}:{https_port}")
            print(f"  • WebSocket: {host_ip_display}:{websocket_port}")
            print("="*70)
            print("🎮 Mode: SO100 Single Arm Control")
            print("="*70 + "\n")
            
            return True
        except Exception as e:
            logger.error(f"❌ Failed to initialize servers: {e}")
            traceback.print_exc()
            return False
    
    async def monitor_commands_async(self):
        """Monitor VR commands asynchronously - right arm only"""
        logger.info("🔄 VR command monitoring started")
        while self.is_running:
            try:
                # Non-blocking get with timeout
                goal = await asyncio.wait_for(self.command_queue.get(), timeout=0.1)
                # Only process RIGHT ARM commands
                if hasattr(goal, 'arm') and goal.arm == "right":
                    with self._goal_lock:
                        self.right_goal = goal
                        self.last_update_time = time.time()
                        logger.debug(f"📍 VR update received")
            except asyncio.TimeoutError:
                continue
            except Exception as e:
                logger.error(f"❌ Command processing error: {e}")
                await asyncio.sleep(0.01)
    
    async def start_monitoring_async(self):
        """Start monitoring VR input (async version)"""
        try:
            # Start HTTPS web server
            logger.info("🌐 Starting HTTPS web server...")
            await self.https_server.start()
            
            # Start VR WebSocket server
            logger.info("🔌 Starting VR WebSocket server...")
            await self.vr_server.start()
            
            self.is_running = True
            logger.info("✅ VR monitoring started")
            await self.monitor_commands_async()
        except Exception as e:
            logger.error(f"❌ VR monitoring error: {e}")
            traceback.print_exc()
        finally:
            await self.stop_monitoring_async()
    
    async def stop_monitoring_async(self):
        """Stop monitoring (async version)"""
        self.is_running = False
        if self.https_server:
            await self.https_server.stop()
        if self.vr_server:
            await self.vr_server.stop()
        logger.info("✅ VR Monitor stopped")
    
    def get_right_goal_nowait(self):
        """Get current right arm goal"""
        with self._goal_lock:
            return self.right_goal
    
    def is_connected(self):
        """Check if VR client is connected"""
        if self.vr_server:
            return len(self.vr_server.clients) > 0
        return False

print("✅ VR Monitor class defined")

✅ VR Monitor class defined


In [4]:
# SO100 VR Controller Class
class SO100VRController:
    """SO100 arm VR teleoperation control"""
    
    def __init__(self, kp=0.5):
        self.kp = kp
        self.current_x = 0.1629
        self.current_y = 0.1131
        self.pitch = 0.0
        
        self.prev_vr_pos = None
        self.prev_wrist_flex = None
        self.prev_wrist_roll = None
        
        # Joint calibration coefficients
        self.joint_calibration = {
            'shoulder_pan': [6.0, 1.0],
            'shoulder_lift': [2.0, 0.97],
            'elbow_flex': [0.0, 1.05],
            'wrist_flex': [0.0, 0.94],
            'wrist_roll': [0.0, 0.5],
            'gripper': [0.0, 1.0],
        }
        
        self.target_positions = {
            "shoulder_pan": 0.0,
            "shoulder_lift": 0.0,
            "elbow_flex": 0.0,
            "wrist_flex": 0.0,
            "wrist_roll": 0.0,
            "gripper": 0.0,
        }

    def apply_joint_calibration(self, joint_name, raw_position):
        """Apply joint calibration coefficients"""
        if joint_name in self.joint_calibration:
            offset, scale = self.joint_calibration[joint_name]
            return (raw_position - offset) * scale
        return raw_position

    def inverse_kinematics(self, x, y, l1=0.1159, l2=0.1350):
        """Calculate inverse kinematics for 2-link arm"""
        theta1_offset = math.atan2(0.028, 0.11257)
        theta2_offset = math.atan2(0.0052, 0.1349) + theta1_offset
        
        r = math.sqrt(x**2 + y**2)
        r_max = l1 + l2
        
        if r > r_max:
            scale_factor = r_max / r
            x *= scale_factor
            y *= scale_factor
            r = r_max
        
        r_min = abs(l1 - l2)
        if r < r_min and r > 0:
            scale_factor = r_min / r
            x *= scale_factor
            y *= scale_factor
            r = r_min
        
        cos_theta2 = -(r**2 - l1**2 - l2**2) / (2 * l1 * l2)
        cos_theta2 = max(-1.0, min(1.0, cos_theta2))
        
        theta2 = math.pi - math.acos(cos_theta2)
        
        beta = math.atan2(y, x)
        gamma = math.atan2(l2 * math.sin(theta2), l1 + l2 * math.cos(theta2))
        theta1 = beta + gamma
        
        joint2 = theta1 + theta1_offset
        joint3 = theta2 + theta2_offset
        
        joint2 = max(-0.1, min(3.45, joint2))
        joint3 = max(-0.2, min(math.pi, joint3))
        
        joint2_deg = math.degrees(joint2)
        joint3_deg = math.degrees(joint3)
        
        joint2_deg = 90 - joint2_deg
        joint3_deg = joint3_deg - 90
        
        return joint2_deg, joint3_deg

    def move_to_zero_position(self, robot, duration=3.0):
        """Move arm to zero position using P control"""
        logger.info("[SO100] Moving to Zero Position...")
        
        zero_positions = {
            'shoulder_pan': 0.0,
            'shoulder_lift': 0.0,
            'elbow_flex': 0.0,
            'wrist_flex': 0.0,
            'wrist_roll': 0.0,
            'gripper': 0.0
        }
        
        control_freq = 50
        total_steps = int(duration * control_freq)
        step_time = 1.0 / control_freq
        
        for step in range(total_steps):
            current_obs = robot.get_observation()
            current_positions = {}
            
            for key, value in current_obs.items():
                if key.endswith('.pos'):
                    motor_name = key.removesuffix('.pos')
                    calibrated_value = self.apply_joint_calibration(motor_name, value)
                    current_positions[motor_name] = calibrated_value
            
            robot_action = {}
            for joint_name, target_pos in zero_positions.items():
                if joint_name in current_positions:
                    current_pos = current_positions[joint_name]
                    error = target_pos - current_pos
                    control_output = self.kp * error
                    new_position = current_pos + control_output
                    robot_action[f"{joint_name}.pos"] = new_position
            
            if robot_action:
                robot.send_action(robot_action)
            
            if step % (control_freq // 2) == 0:
                progress = (step / total_steps) * 100
                logger.info(f"Moving to zero position: {progress:.1f}%")
            
            time.sleep(step_time)
        
        logger.info("✅ Robot moved to zero position")

    def handle_vr_input(self, vr_goal):
        """Handle VR input with delta action control - incremental position updates"""
        if vr_goal is None or not hasattr(vr_goal, 'target_position') or vr_goal.target_position is None:
            return
        
        current_vr_pos = vr_goal.target_position
        
        # Initialize previous VR position if not set
        if self.prev_vr_pos is None:
            self.prev_vr_pos = current_vr_pos
            return  # Skip first frame to establish baseline
        
        # Calculate relative change (delta) from previous frame
        vr_x = (current_vr_pos[0] - self.prev_vr_pos[0]) * 220  # Scale for shoulder pan
        vr_y = (current_vr_pos[1] - self.prev_vr_pos[1]) * 70
        vr_z = (current_vr_pos[2] - self.prev_vr_pos[2]) * 70
        
        # Update previous position for next frame
        self.prev_vr_pos = current_vr_pos
        
        # Delta control parameters - adjust these for sensitivity
        pos_scale = 0.01  # Position sensitivity scaling
        angle_scale = 4.0  # Angle sensitivity scaling
        delta_limit = 0.01  # Maximum delta per update (meters)
        angle_limit = 8.0  # Maximum angle delta per update (degrees)
        
        delta_x = vr_x * pos_scale
        delta_y = vr_y * pos_scale
        delta_z = vr_z * pos_scale
        
        # Limit delta values to prevent sudden movements
        delta_x = max(-delta_limit, min(delta_limit, delta_x))
        delta_y = max(-delta_limit, min(delta_limit, delta_y))
        delta_z = max(-delta_limit, min(delta_limit, delta_z))
        
        # Update end-effector position incrementally (relative position)
        self.current_x += -delta_z  # VR Z maps to robot x
        self.current_y += delta_y   # VR Y maps to robot y
        
        # Handle wrist angles with delta control - use relative changes
        if hasattr(vr_goal, 'wrist_flex_deg') and vr_goal.wrist_flex_deg is not None:
            # Initialize previous wrist_flex if not set
            if self.prev_wrist_flex is None:
                self.prev_wrist_flex = vr_goal.wrist_flex_deg
            else:
                # Calculate relative change from previous frame
                delta_pitch = (vr_goal.wrist_flex_deg - self.prev_wrist_flex) * angle_scale
                delta_pitch = max(-angle_limit, min(angle_limit, delta_pitch))
                self.pitch += delta_pitch
                self.pitch = max(-90, min(90, self.pitch))  # Limit pitch range
                
                # Update previous value for next frame
                self.prev_wrist_flex = vr_goal.wrist_flex_deg
        
        if hasattr(vr_goal, 'wrist_roll_deg') and vr_goal.wrist_roll_deg is not None:
            # Initialize previous wrist_roll if not set
            if self.prev_wrist_roll is None:
                self.prev_wrist_roll = vr_goal.wrist_roll_deg
            else:
                delta_roll = (vr_goal.wrist_roll_deg - self.prev_wrist_roll) * angle_scale
                delta_roll = max(-angle_limit, min(angle_limit, delta_roll))
                
                current_roll = self.target_positions.get("wrist_roll", 0.0)
                new_roll = current_roll + delta_roll
                new_roll = max(-90, min(90, new_roll))  # Limit roll range
                self.target_positions["wrist_roll"] = new_roll
                
                # Update previous value for next frame
                self.prev_wrist_roll = vr_goal.wrist_roll_deg
        
        # VR X axis controls shoulder_pan joint (delta control)
        if abs(delta_x) > 0.001:  # Only update if significant movement
            x_scale = 200.0  # Scaling factor for delta control
            delta_pan = delta_x * x_scale
            delta_pan = max(-angle_limit, min(angle_limit, delta_pan))
            current_pan = self.target_positions.get("shoulder_pan", 0.0)
            new_pan = current_pan + delta_pan
            new_pan = max(-180, min(180, new_pan))  # Limit pan range
            self.target_positions["shoulder_pan"] = new_pan
        
        # Inverse kinematics for relative position control
        try:
            joint2_target, joint3_target = self.inverse_kinematics(self.current_x, self.current_y)
            # Smooth transition to new joint positions
            alpha = 0.1  # Smoothing factor (lower = smoother)
            self.target_positions["shoulder_lift"] = (
                (1 - alpha) * self.target_positions.get("shoulder_lift", 0.0) + 
                alpha * joint2_target
            )
            self.target_positions["elbow_flex"] = (
                (1 - alpha) * self.target_positions.get("elbow_flex", 0.0) + 
                alpha * joint3_target
            )
        except Exception as e:
            logger.debug(f"IK calculation error: {e}")
        
        # Calculate wrist_flex to maintain end-effector orientation
        self.target_positions["wrist_flex"] = (
            -self.target_positions["shoulder_lift"] - 
            self.target_positions["elbow_flex"] + 
            self.pitch
        )
        
    def gripper_control(self, vr_goal):
        """Control gripper based on VR trigger input"""
        if hasattr(vr_goal, 'metadata') and vr_goal.metadata.get('trigger', 0) > 0.5:
            self.target_positions["gripper"] = 0.0
        else:
            self.target_positions["gripper"] = 45.0

    def p_control_action(self, robot):
        """Generate P-control action"""
        obs = robot.get_observation()
        action = {}
        
        for joint_name, target_pos in self.target_positions.items():
            current_key = f"{joint_name}.pos"
            if current_key in obs:
                current = obs[current_key]
                # Don't apply calibration when calculating error
                error = target_pos - current
                control = self.kp * error
                action[f"{joint_name}.pos"] = current + control
        
        return action

print("✅ SO100 VR Controller class defined")

✅ SO100 VR Controller class defined


---

## 🤖 SECTION 1: 로봇 연결

**SO100 로봇을 연결하고 초기화합니다.**

✅ 확인사항:
- 로봇이 전원 ON되어 있나요?
- USB 포트가 연결되어 있나요? (기본값: /dev/ttyACM0)

In [7]:
# Connect to SO100 Robot
print("🔄 Connecting to SO100...")

import yaml
from pathlib import Path

config_path = Path("/home/choyunsang/lerobot/config.yaml")

if not config_path.exists():
        print(f"❌ Config file not found: {config_path}")
        print("   Using default values instead...")
        # Fallback to defaults
        port = "/dev/ttyACM1"
        kp = 0.5

else:
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    
    # Extract dataset configuration
    port = config.get('robot', {}).get('port', "/dev/ttyACM0")
    kp = config.get('robot', {}).get('kp', 0.5)

try:
    from lerobot.robots.so_follower.so_follower import SO100Follower
    from lerobot.robots.so_follower.config_so_follower import SO100FollowerConfig
    
    # Use default port or specify manually
    
    robot_config = SO100FollowerConfig(port=port)
    robot = SO100Follower(robot_config)
    
    robot.connect()
    logger.info(f"✅ Robot connected successfully on {port}")
    
    if robot.is_calibrated:
        logger.info("✅ Robot is calibrated")
    else:
        logger.warning("⚠️  Robot needs calibration")
    
    # Store robot reference
    globals_dict['robot'] = robot
    
    # Initialize VR controller
    controller = SO100VRController(kp=kp)
    
    # Move to zero position
    logger.info("🎯 Moving to zero position...")
    controller.move_to_zero_position(robot)
    
    globals_dict['controller'] = controller
    
    print("\n" + "="*70)
    print("✅ ROBOT CONNECTED AND READY")
    print("="*70)
    print("Next Step: Run SECTION 2 to setup MetaQuest connection")
    
except Exception as e:
    logger.error(f"❌ Robot connection failed: {e}")
    traceback.print_exc()

2026-05-20 19:23:05,693 - lerobot.robots.so_follower.so_follower - INFO - None SOFollower connected.
2026-05-20 19:23:05,693 - __main__ - INFO - ✅ Robot connected successfully on /dev/ttyACM0
2026-05-20 19:23:05,699 - __main__ - INFO - ✅ Robot is calibrated
2026-05-20 19:23:05,699 - __main__ - INFO - 🎯 Moving to zero position...
2026-05-20 19:23:05,700 - __main__ - INFO - [SO100] Moving to Zero Position...
2026-05-20 19:23:05,701 - __main__ - INFO - Moving to zero position: 0.0%


🔄 Connecting to SO100...


2026-05-20 19:23:06,255 - __main__ - INFO - Moving to zero position: 16.7%
2026-05-20 19:23:06,798 - __main__ - INFO - Moving to zero position: 33.3%
2026-05-20 19:23:07,348 - __main__ - INFO - Moving to zero position: 50.0%
2026-05-20 19:23:07,897 - __main__ - INFO - Moving to zero position: 66.7%
2026-05-20 19:23:08,444 - __main__ - INFO - Moving to zero position: 83.3%
2026-05-20 19:23:08,990 - __main__ - INFO - ✅ Robot moved to zero position



✅ ROBOT CONNECTED AND READY
Next Step: Run SECTION 2 to setup MetaQuest connection


In [ ]:
# Initialize Dataset for VR Data Collection
print("📊 Initializing Dataset from config.yaml...")

try:
    import yaml
    from pathlib import Path
    from lerobot.datasets.lerobot_dataset import LeRobotDataset
    from lerobot.utils.feature_utils import build_dataset_frame, hw_to_dataset_features
    
    # Load configuration from config.yaml
    config_path = Path("/home/choyunsang/lerobot/config.yaml")
    
    if not config_path.exists():
        print(f"❌ Config file not found: {config_path}")
        print("   Using default values instead...")
        # Fallback to defaults
        dataset_repo_id = "your_username/so100_vr_teleop_data"
        dataset_root = "./datasets"
        dataset_fps = 30
        single_task = "VR teleoperation control"
        num_episodes = 10
    else:
        with open(config_path, 'r') as f:
            config = yaml.safe_load(f)
        
        # Extract dataset configuration
        dataset_config = config.get('dataset', {})
        dataset_repo_id = dataset_config.get('repo_id', 'so100_vr_record')
        task_name = dataset_config.get('task_name', 'SO100_VR_teleoperation')
        dataset_root = dataset_config.get('root', './datasets')
        dataset_fps = dataset_config.get('fps', 30)
        single_task = dataset_config.get('single_task', 'VR teleoperation task')
        num_episodes = dataset_config.get('num_episodes', 5)
        
        # Extract image writer configuration
        image_writer_config = config.get('image_writer', {})
        num_image_processes = image_writer_config.get('num_processes', 0)
        num_image_threads = image_writer_config.get('threads_per_camera', 4)
        batch_encoding_size = image_writer_config.get('batch_encoding_size', 1)
        
        print(f"✅ Configuration loaded from: {config_path}")
    
    # Create dataset features from robot action/observation features
    if globals_dict['robot']:
        robot = globals_dict['robot']
        
        # Generate dataset features (record.py line 454-456)
        action_features = hw_to_dataset_features(robot.action_features, "action", use_video=False)
        obs_features = hw_to_dataset_features(robot.observation_features, "observation", use_video=False)
        dataset_features = {**action_features, **obs_features}
        
        # Try to create new dataset, or resume existing if it already exists
        dataset = None
        dataset_path = Path(dataset_root)
        dataset_path = dataset_path.joinpath(task_name)

        print(f"📁 Dataset path: {dataset_path}")
        
        if not dataset_path.exists():
            print(f"✨ Creating new dataset...")
            dataset = LeRobotDataset.create(
                repo_id=dataset_repo_id,
                fps=dataset_fps,
                root=dataset_path,
                robot_type=robot.name,
                features=dataset_features,
                use_videos=False,  # Set to True if you want video encoding
                image_writer_processes=num_image_processes,
                image_writer_threads=num_image_threads * len(robot.cameras) if hasattr(robot, 'cameras') else 4,
            )
            print(f"✅ Created new dataset")
            created_new = True
            
        else:
            print(f"📂 Dataset already exists. Resuming to add new data...")
            try:
                dataset = LeRobotDataset.resume(
                    repo_id=dataset_repo_id,
                    root=dataset_path,
                    batch_encoding_size=batch_encoding_size,
                    image_writer_processes=num_image_processes,
                    image_writer_threads=num_image_threads * len(robot.cameras) if hasattr(robot, 'cameras') else 4,
                )
                print(f"✅ Resumed existing dataset")
                created_new = False
                
            except Exception as resume_error:
                print(f"⚠️  Error resuming dataset: {resume_error}")
                print(f"   Dataset path: {dataset_path}")
                print(f"   Please verify the dataset exists and is not corrupted")
                raise
        
        globals_dict['dataset'] = dataset
        globals_dict['dataset_repo_id'] = dataset_repo_id
        globals_dict['dataset_features'] = dataset_features
        globals_dict['single_task'] = single_task
        globals_dict['num_episodes'] = num_episodes
        
        print("\n" + "="*70)
        print("✅ DATASET INITIALIZED")
        print("="*70)
        print(f"📁 Repo ID: {dataset_repo_id}")
        print(f"📂 Root: {dataset_root}")
        print(f"📹 FPS: {dataset_fps}")
        print(f"📋 Task: {single_task}")
        print(f"🎬 Target Episodes: {num_episodes}")
        print(f"📝 Image Writer Processes: {num_image_processes}")
        print(f"🧵 Image Writer Threads/Camera: {num_image_threads}")
        print(f"🔑 Features: {len(dataset_features)} total")
        print(f"📊 Current Episodes: {dataset.num_episodes}")
        print(f"🎬 Current Frames: {dataset.num_frames}")
        status = "📌 NEW" if created_new else "➕ RESUMED"
        print(f"{status}")
        print("="*70)
        print("\n📋 Configuration Source: config.yaml")
        print("   (Edit /home/choyunsang/lerobot/config.yaml to change settings)")
        print("\nNext: Run SECTION 2 to setup MetaQuest connection")
    else:
        print("❌ Robot not connected. Run SECTION 1 first!")

except Exception as e:
    logger.error(f"❌ Dataset initialization failed: {e}")
    traceback.print_exc()


SyntaxError: '[' was never closed (416874855.py, line 45)

---

## 📱 SECTION 2: MetaQuest 연결

**VR 서버를 시작하고 MetaQuest 헤드셋을 연결합니다.**

✅ 이 셀을 실행하면:
1. **HTTPS 웹 UI 서버** 시작 (포트 8443) - 웹 페이지 제공
2. **WebSocket 서버** 시작 (포트 8442) - VR 데이터 수신
3. **MetaQuest 접속 주소 표시** - 복사해서 헤드셋 브라우저에 입력
   - 🔗 URL 형식: `https://[IP]:[HTTPS_PORT]`
   - 예: `https://192.168.1.100:8443`

⚠️ **중요 - SSL 인증서 경고:**
- MetaQuest 브라우저에서 **"연결이 안전하지 않습니다"** 경고가 나올 수 있습니다
- 이는 **자체 서명된 인증서를 사용**하기 때문입니다 (정상입니다)
- **해결 방법:**
  - 🔐 "연결이 안전하지 않습니다" 화면 보임
  - ➡️ **"고급" 또는 "Advanced"** 클릭
  - ➡️ **"계속" 또는 "Continue"** 클릭
  - ➡️ 웹 페이지 로드 완료!

⏰ 주의: 이 셀은 **계속 실행 상태를 유지**합니다
- 헤드셋에서 접속 후, 다음 셀(SECTION 3)을 실행하세요
- 🟢 메시지에서 "MetaQuest Connected" 표시되면 준비 완료

In [ ]:
# Setup VR Monitor with proper asyncio event loop management
print("⏳ Starting VR Monitor (HTTPS + WebSocket)...")

try:
    # Stop previous event loop if it exists
    if globals_dict['event_loop_thread'] is not None:
        globals_dict['event_loop_thread'].stop()
        time.sleep(0.5)
    
    # Create and start a new event loop thread
    event_loop_thread = AsyncEventLoopThread()
    event_loop_thread.start()
    event_loop_thread.ready.wait(timeout=5)  # Wait for loop to be ready
    
    if event_loop_thread.loop is None:
        print("❌ Failed to create asyncio event loop")
    else:
        logger.info("✅ Asyncio event loop thread started")
        globals_dict['event_loop_thread'] = event_loop_thread
        globals_dict['event_loop'] = event_loop_thread.loop
        
        # Initialize VR monitor
        vr_monitor = VRControlMonitor()
        
        if not vr_monitor.initialize():
            print("❌ VR Monitor initialization failed")
        else:
            print("\n" + "="*70)
            print("✅ VR SERVERS READY")
            print("="*70)
            
            # Start VR monitoring in the event loop thread
            future = event_loop_thread.run_coroutine(vr_monitor.start_monitoring_async())
            
            globals_dict['vr_monitor'] = vr_monitor
            
            # Wait a moment for servers to start
            time.sleep(1)
            
            print("\n📋 CONNECTION GUIDE:")
            print("1. ⬆️  COPY the HTTPS URL from above")
            print("   (Should start with 'https://' NOT 'http://')")
            print("")
            print("2. 🥽 Open your MetaQuest headset browser")
            print("")
            print("3. 📍 Paste the URL into the address bar")
            print("")
            print("4. ⚠️  You may see 'Not secure' warning:")
            print("   → Look for 'Advanced' or 'More options' button")
            print("   → Click it to see more options")
            print("   → Click 'Proceed' or 'Continue anyway'")
            print("   → Wait for page to load")
            print("")
            print("5. ✅ Once page loads, run SECTION 3 to start control")
            print("\n⏳ Waiting for MetaQuest connection...")
            print("(This cell keeps running, that's normal)\n")
            
            # Wait for connection
            wait_time = 0
            connected = False
            while not connected and wait_time < 60:
                if vr_monitor.is_connected():
                    connected = True
                    break
                
                time.sleep(1)
                wait_time += 1
                
                if wait_time % 5 == 0:
                    print(f"🔄 Waiting for MetaQuest... ({wait_time}s)")
            
            if connected:
                print("\n" + "="*70)
                print("🎉 MetaQuest Connected!")
                print("="*70)
                print("✅ Ready for control - Run SECTION 3")
            else:
                print("\n" + "="*70)
                print("⏳ Still waiting for connection...")
                print("="*70)
                print("You can still proceed to SECTION 3")
                print("(Connection may happen later)")
        
except Exception as e:
    logger.error(f"❌ VR Monitor error: {e}")
    traceback.print_exc()

2026-05-18 21:12:41,677 - __main__ - INFO - ✅ Asyncio event loop thread started
2026-05-18 21:12:41,678 - __main__ - INFO - 🔧 Initializing VR Monitor (SO100)...
2026-05-18 21:12:41,802 - __main__ - INFO - ✅ XLeVR modules imported
2026-05-18 21:12:41,803 - __main__ - INFO - ✅ VR WebSocket server created
2026-05-18 21:12:41,804 - __main__ - INFO - ✅ HTTPS web server created
2026-05-18 21:12:41,805 - __main__ - INFO - 🌐 Starting HTTPS web server...
2026-05-18 21:12:41,806 - __main__ - INFO - ✅ HTTPS server started on 0.0.0.0:8443
2026-05-18 21:12:41,807 - __main__ - INFO - 🔌 Starting VR WebSocket server...
2026-05-18 21:12:41,807 - xlevr.inputs.vr_ws_server - INFO - SSL certificate and key loaded successfully for WebSocket server
2026-05-18 21:12:41,816 - websockets.server - INFO - server listening on 0.0.0.0:8442
2026-05-18 21:12:41,817 - xlevr.inputs.vr_ws_server - INFO - VR WebSocket server running on wss://0.0.0.0:8442
2026-05-18 21:12:41,817 - __main__ - INFO - ✅ VR monitoring starte

⏳ Starting VR Monitor (HTTPS + WebSocket)...

📱 MetaQuest VR Headset - COPY THIS URL:

   https://192.168.0.44:8443

Connection Details:
  • HTTPS Web UI: 192.168.0.44:8443
  • WebSocket: 192.168.0.44:8442
🎮 Mode: SO100 Single Arm Control


✅ VR SERVERS READY

📋 CONNECTION GUIDE:
1. ⬆️  COPY the HTTPS URL from above
   (Should start with 'https://' NOT 'http://')

2. 🥽 Open your MetaQuest headset browser

3. 📍 Paste the URL into the address bar

4. ⚠️  You may see 'Not secure' warning:
   → Look for 'Advanced' or 'More options' button
   → Click it to see more options
   → Click 'Proceed' or 'Continue anyway'
   → Wait for page to load

5. ✅ Once page loads, run SECTION 3 to start control

⏳ Waiting for MetaQuest connection...
(This cell keeps running, that's normal)



2026-05-18 21:12:45,857 - websockets.server - INFO - connection open
2026-05-18 21:12:45,858 - xlevr.inputs.vr_ws_server - INFO - VR client connected: ('192.168.0.65', 46010)
2026-05-18 21:12:45,903 - xlevr.inputs.vr_ws_server - INFO - 🎯 LEFT auto-activated - controlling left arm
2026-05-18 21:12:45,906 - xlevr.inputs.vr_ws_server - INFO - 🎯 RIGHT auto-activated - controlling right arm


[VR_WS] Headset - Position: [0.000, 0.000, 0.000], Rotation: [0.0, 0.0, 0.0]
[VR_WS] Headset - Position: [0.000, 0.000, 0.000], Rotation: [0.0, 0.0, 0.0]
[VR_WS] Headset - Position: [0.000, 0.000, 0.000], Rotation: [0.0, 0.0, 0.0]
[VR_WS] Headset - Position: [0.000, 0.000, 0.000], Rotation: [0.0, 0.0, 0.0]
[VR_WS] Headset - Position: [0.000, 0.000, 0.000], Rotation: [0.0, 0.0, 0.0]
[VR_WS] Headset - Position: [0.000, 0.000, 0.000], Rotation: [0.0, 0.0, 0.0]
[VR_WS] Headset - Position: [0.000, 0.000, 0.000], Rotation: [0.0, 0.0, 0.0]
[VR_WS] Headset - Position: [0.000, 0.000, 0.000], Rotation: [0.0, 0.0, 0.0]
[VR_WS] Headset - Position: [0.000, 0.000, 0.000], Rotation: [0.0, 0.0, 0.0]
[VR_WS] Headset - Position: [0.000, 0.000, 0.000], Rotation: [0.0, 0.0, 0.0]
[VR_WS] Headset - Position: [0.000, 0.000, 0.000], Rotation: [0.0, 0.0, 0.0]
[VR_WS] Headset - Position: [0.000, 0.000, 0.000], Rotation: [0.0, 0.0, 0.0]
[VR_WS] Headset - Position: [0.000, 0.000, 0.000], Rotation: [0.0, 0.0, 0.0]

[VR_WS] Headset - Position: [0.000, 0.000, 0.000], Rotation: [0.0, 0.0, 0.0]
[VR_WS] Headset - Position: [0.000, 0.000, 0.000], Rotation: [0.0, 0.0, 0.0]
[VR_WS] Headset - Position: [0.000, 0.000, 0.000], Rotation: [0.0, 0.0, 0.0]
[VR_WS] Headset - Position: [0.000, 0.000, 0.000], Rotation: [0.0, 0.0, 0.0]
[VR_WS] Headset - Position: [0.000, 0.000, 0.000], Rotation: [0.0, 0.0, 0.0]
[VR_WS] Headset - Position: [0.000, 0.000, 0.000], Rotation: [0.0, 0.0, 0.0]
[VR_WS] Headset - Position: [0.000, 0.000, 0.000], Rotation: [0.0, 0.0, 0.0]
[VR_WS] Headset - Position: [0.000, 0.000, 0.000], Rotation: [0.0, 0.0, 0.0]
[VR_WS] Headset - Position: [0.000, 0.000, 0.000], Rotation: [0.0, 0.0, 0.0]
[VR_WS] Headset - Position: [0.000, 0.000, 0.000], Rotation: [0.0, 0.0, 0.0]
[VR_WS] Headset - Position: [0.000, 0.000, 0.000], Rotation: [0.0, 0.0, 0.0]
[VR_WS] Headset - Position: [0.000, 0.000, 0.000], Rotation: [0.0, 0.0, 0.0]
[VR_WS] Headset - Position: [0.000, 0.000, 0.000], Rotation: [0.0, 0.0, 0.0]

---

## 🎮 SECTION 3: VR 제어 시작

**MetaQuest 컨트롤러로 SO100 로봇을 제어합니다.**

✅ 사전 요구사항:
- ✔️ SECTION 1에서 로봇 연결 완료
- ✔️ SECTION 2에서 MetaQuest 접속 완료
- ✔️ MetaQuest 헤드셋이 연결되어 있음

🎮 컨트롤러 사용법:
- **위치 이동**: 중지 버튼 누른 채로 컨트롤러 위치 이동 → 팔의 엔드이펙터 위치 변경
- **손목 회전**: 컨트롤러 손목 각도 → 손목 조정
- **그리퍼**: 트리거 버튼 → 그리퍼 개폐

🔄 **실시간 모니터링**:
- VR 입력과 로봇 제어가 동시에 실행됩니다
- 매 2초마다 현재 상태가 표시됩니다 (루프 주파수, VR 업데이트 빈도, 엔드이펙터 위치)
- 실제 VR 조종 데이터가 계속 받아집니다

⏹️ 종료: 셀 실행 중지 (Stop 버튼) → 로봇 안전 정지

In [8]:
# VR Metadata Configuration - Modify metadata behavior
print("⚙️  VR Metadata Configuration (Squeeze Button Mode)")
print("="*70)

# Configuration for metadata handling
metadata_config = {
    'override_buttons': False,           # Override buttons with custom value
    'buttons_squeeze_value': True,       # Custom squeeze button state
    'add_buttons_field': True,           # Add buttons field to metadata
}

print("\n📝 Current Configuration:")
print("   override_buttons: Use custom squeeze button value")
print(f"     → {metadata_config['override_buttons']} (Current: use VR data)")
print("   buttons_squeeze_value: Custom squeeze state")
print(f"     → {metadata_config['buttons_squeeze_value']}")
print("   add_buttons_field: Add buttons field to metadata")
print(f"     → {metadata_config['add_buttons_field']}")

print("\n💡 To modify:")
print("   1. Change values above")
print("   2. Run this cell again")
print("   3. Then run SECTION 3")

print("\n🎮 Control Mode: SQUEEZE BUTTON ONLY")
print("   (trigger is NOT used for control)")
print("\n" + "="*70)

# Store config
globals_dict['metadata_config'] = metadata_config
print("✅ Metadata config ready (stored in globals_dict)")


⚙️  VR Metadata Configuration (Squeeze Button Mode)

📝 Current Configuration:
   override_buttons: Use custom squeeze button value
     → False (Current: use VR data)
   buttons_squeeze_value: Custom squeeze state
     → True
   add_buttons_field: Add buttons field to metadata
     → True

💡 To modify:
   1. Change values above
   2. Run this cell again
   3. Then run SECTION 3

🎮 Control Mode: SQUEEZE BUTTON ONLY
   (trigger is NOT used for control)

✅ Metadata config ready (stored in globals_dict)


In [9]:
# Main VR Control Loop with Data Collection
print("🚀 Starting VR Control Loop with Data Collection...")
print("💡 Press Stop button to exit\n")
print("🎮 Control Mode: TRIGGER/SQUEEZE BUTTON REQUIRED")
print("   (Release button to stop control)\n")

robot = globals_dict.get('robot')
controller = globals_dict.get('controller')
vr_monitor = globals_dict.get('vr_monitor')
dataset = globals_dict.get('dataset')
dataset_features = globals_dict.get('dataset_features')
single_task = globals_dict.get('single_task')
num_episodes = globals_dict.get('num_episodes', 10)
metadata_config = globals_dict.get('metadata_config', {})

if not all([robot, controller, vr_monitor]):
    print("❌ Error: Missing robot, controller, or VR monitor")
    print("   Run SECTION 1, SECTION 1B, and SECTION 2 first!")
else:
    try:
        loop_count = 0
        start_time = time.time()
        vr_update_count = 0
        last_print_time = time.time()
        episode_count = 0
        frames_in_episode = 0
        squeeze_pressed_prev = False
        debug_mode = True  # Debug flag to print metadata once
        
        print("📊 Monitoring Status:")
        print("-" * 70)
        
        # Control loop
        while episode_count < num_episodes:
            # Get VR input
            right_goal = vr_monitor.get_right_goal_nowait()
            squeeze_pressed = False
            
            # Check if squeeze button is pressed
            if right_goal is not None:
                # Debug: Print metadata structure once
                if debug_mode:
                    print("\n🔍 DEBUG - Metadata structure:")
                    print(f"   hasattr metadata: {hasattr(right_goal, 'metadata')}")
                    if hasattr(right_goal, 'metadata'):
                        print(f"   metadata type: {type(right_goal.metadata)}")
                        print(f"   metadata keys: {right_goal.metadata.keys() if right_goal.metadata else 'None'}")
                        if right_goal.metadata:
                            for key, value in right_goal.metadata.items():
                                if key != 'vr_position':  # Skip large arrays
                                    print(f"     - {key}: {value}")
                    debug_mode = False
                
                # Modify metadata if configured
                if hasattr(right_goal, 'metadata') and right_goal.metadata:
                    # Add buttons field if configured
                    if metadata_config.get('add_buttons_field', True):
                        if 'buttons' not in right_goal.metadata:
                            right_goal.metadata['buttons'] = {}
                    
                    # Override trigger_active if configured
                    if metadata_config.get('override_trigger', False):
                        right_goal.metadata['trigger_active'] = metadata_config.get('trigger_active_value', False)
                        logger.info(f"🔧 Overriding trigger_active → {right_goal.metadata['trigger_active']}")
                    
                    # Override buttons squeeze if configured
                    if metadata_config.get('override_buttons', False):
                        right_goal.metadata['buttons']['squeeze'] = metadata_config.get('buttons_squeeze_value', True)
                        logger.info(f"🔧 Overriding buttons.squeeze → {right_goal.metadata['buttons']['squeeze']}")
                
                # Try to extract squeeze button state
                if hasattr(right_goal, 'metadata') and right_goal.metadata:
                    buttons = right_goal.metadata.get('buttons', {})
                    trigger_active = right_goal.metadata.get('trigger_active', False)
                    
                    # Priority: buttons.squeeze > trigger_active
                    if isinstance(buttons, dict):
                        squeeze_pressed = buttons.get('squeeze', False)
                    
                    if not squeeze_pressed:
                        squeeze_pressed = trigger_active
                    
                    # Log when button state changes
                    if squeeze_pressed and not squeeze_pressed_prev:
                        logger.info("🔘 Control button pressed - Control ACTIVE")
                        squeeze_pressed_prev = True
                    elif not squeeze_pressed and squeeze_pressed_prev:
                        logger.info("🔘 Control button released - Control INACTIVE")
                        squeeze_pressed_prev = False
                
                controller.gripper_control(right_goal)

                # Only process VR input if squeeze button is pressed
                if squeeze_pressed:
                    controller.handle_vr_input(right_goal)
                    vr_update_count += 1
            
            # Get observation from robot (record.py line 352)
            try:
                observation = robot.get_observation()
            except TimeoutError as e:
                logger.warning(f"Camera timeout: {e}. Skipping this frame.")
                continue
            
            # Generate and send action to robot
            action = controller.p_control_action(robot)
            sent_action = robot.send_action(action)
            
            # Collect data if dataset exists (record.py line 426-431)
            if dataset is not None:
                # Build observation frame
                observation_frame = build_dataset_frame(
                    dataset_features, 
                    observation, 
                    prefix="observation"
                )
                
                # Build action frame
                action_frame = build_dataset_frame(
                    dataset_features,
                    sent_action,
                    prefix="action"
                )
                
                # Combine frames with task info
                frame = {**observation_frame, **action_frame, "task": single_task}
                
                # Add frame to dataset
                dataset.add_frame(frame)
                frames_in_episode += 1
            
            loop_count += 1
            
            # Status update every 2 seconds
            current_time = time.time()
            if current_time - last_print_time >= 2.0:
                elapsed = current_time - start_time
                vr_status = "📍 VR Connected" if vr_monitor.is_connected() else "⏳ Waiting VR"
                vr_freq = vr_update_count / max(elapsed, 1)
                squeeze_status = "🔘 ACTIVE" if squeeze_pressed else "⭕ INACTIVE"
                
                print(f"⏱️  {elapsed:6.1f}s | Ep: {episode_count+1}/{num_episodes} | Frames: {frames_in_episode:5d} | "
                      f"Loop: {loop_count:6d} ({loop_count/max(elapsed,1):.0f}Hz) | "
                      f"VR: {vr_freq:.1f}Hz | {squeeze_status} | {vr_status}")
                
                last_print_time = current_time
            
            # Control loop frequency (50Hz)
            time.sleep(0.02)
        
        # Save episode and increment counter when finishing
        if dataset is not None and frames_in_episode > 0:
            logger.info(f"💾 Saving episode {episode_count + 1}/{num_episodes}...")
            dataset.save_episode()
            episode_count += 1
            frames_in_episode = 0
            
            print(f"\n✅ Episode {episode_count} saved ({frames_in_episode} frames)")
            
    except KeyboardInterrupt:
        print("\n⏹️  Stopping control loop...")
        # Save current episode if it has data
        if dataset is not None and frames_in_episode > 0:
            logger.info("💾 Saving current episode...")
            dataset.save_episode()
            episode_count += 1
            print(f"✅ Current episode saved ({frames_in_episode} frames)")
    except Exception as e:
        print(f"\n❌ Control error: {e}")
        traceback.print_exc()
    finally:
        print("✅ VR Control Loop ended")
        if dataset is not None:
            dataset.finalize()
            print(f"\n📊 Dataset Summary:")
            print(f"   Total Episodes: {episode_count}")
            print(f"   Total Frames: {dataset.num_frames}")
            print(f"   Repo ID: {dataset.repo_id}")


🚀 Starting VR Control Loop with Data Collection...
💡 Press Stop button to exit

🎮 Control Mode: TRIGGER/SQUEEZE BUTTON REQUIRED
   (Release button to stop control)

📊 Monitoring Status:
----------------------------------------------------------------------

🔍 DEBUG - Metadata structure:
   hasattr metadata: True
   metadata type: <class 'dict'>
   metadata keys: dict_keys(['source', 'relative_position', 'vr_position', 'scaled_position', 'trigger', 'trigger_active', 'buttons', 'thumbstick'])
     - source: vr_absolute_position
     - relative_position: False
     - scaled_position: [0.48908424377441406, 0.947342038154602, 0.49617671966552734]
     - trigger: 0
     - trigger_active: False
     - buttons: {}
     - thumbstick: {}
⏱️     2.0s | Ep: 1/5 | Frames:    87 | Loop:     87 (43Hz) | VR: 0.0Hz | ⭕ INACTIVE | 📍 VR Connected


2026-05-18 21:12:56,119 - __main__ - INFO - 🔘 Control button pressed - Control ACTIVE


⏱️     4.0s | Ep: 1/5 | Frames:   174 | Loop:    174 (43Hz) | VR: 8.4Hz | 🔘 ACTIVE | 📍 VR Connected


2026-05-18 21:12:57,058 - __main__ - INFO - 🔘 Control button released - Control INACTIVE


⏱️     6.0s | Ep: 1/5 | Frames:   260 | Loop:    260 (43Hz) | VR: 6.6Hz | ⭕ INACTIVE | 📍 VR Connected


2026-05-18 21:12:59,919 - __main__ - INFO - 💾 Saving current episode...



⏹️  Stopping control loop...


Map: 100%|██████████| 303/303 [00:00<00:00, 3229.26 examples/s]

✅ Current episode saved (303 frames)
✅ VR Control Loop ended

📊 Dataset Summary:
   Total Episodes: 1
   Total Frames: 1169
   Repo ID: ranggae/so100_vr_record


---

## 🛑 CLEANUP: 안전하게 종료

**로봇과 VR 연결을 안전하게 해제합니다.**

⚠️ 제어를 종료할 때 항상 이 셀을 실행하세요!

## !!셀 실행 시 로봇팔의 토크가 갑자기 풀리니 주의!!

In [1]:
# Cleanup and Shutdown
print("🔄 Cleaning up...")

robot = globals_dict.get('robot')
vr_monitor = globals_dict.get('vr_monitor')
event_loop_thread = globals_dict.get('event_loop_thread')

try:
    if robot:
        robot.disconnect()
        logger.info("✅ Robot disconnected")
    
    if vr_monitor:
        vr_monitor.is_running = False
        logger.info("✅ VR Monitor stopped")
    
    if event_loop_thread:
        event_loop_thread.stop()
        logger.info("✅ Event loop thread stopped")
    
    # Give threads time to stop
    time.sleep(0.5)
    
    print("\n" + "="*70)
    print("✅ ALL SYSTEMS SAFELY SHUT DOWN")
    print("="*70)
    
except Exception as e:
    logger.error(f"❌ Cleanup error: {e}")
    traceback.print_exc()

# Reset globals
globals_dict['robot'] = None
globals_dict['vr_monitor'] = None
globals_dict['controller'] = None
globals_dict['event_loop_thread'] = None
globals_dict['event_loop'] = None

🔄 Cleaning up...


NameError: name 'globals_dict' is not defined

---

## 📤 DATASET UPLOAD: Hugging Face Hub에 업로드

**수집한 데이터를 Hugging Face Hub에 저장합니다.**

✅ 사전 요구사항:
- 💾 SECTION 3에서 데이터 수집 완료
- 🔑 `huggingface-hub` 패키지 설치 (자동)
- 👤 Hugging Face 계정 로그인
  - 터미널에서: `huggingface-cli login`
  - 또는 `HF_TOKEN` 환경변수 설정

📌 주의:
- 리포지토리가 존재하지 않으면 자동 생성됨
- 기존 데이터가 있으면 새 에피소드가 추가됨 (덮어쓰기 아님)


In [ ]:
# Upload Dataset to Hugging Face Hub
print("📤 Uploading dataset to Hugging Face Hub...")

dataset = globals_dict.get('dataset')
dataset_repo_id = globals_dict.get('dataset_repo_id')

if dataset is None:
    print("❌ No dataset found. Run SECTION 3 first!")
else:
    try:
        print(f"\n📋 Dataset Info:")
        print(f"   Repo ID: {dataset_repo_id}")
        print(f"   Total Episodes: {dataset.num_episodes}")
        print(f"   Total Frames: {dataset.num_frames}")
        print(f"   FPS: {dataset.fps}")
        
        print(f"\n⏳ Uploading to hub...")
        
        # Upload to Hugging Face Hub
        dataset.push_to_hub(
            tags=["so100", "vr-teleop", "teleoperation"],
            private=False  # Set to True if you want private repo
        )
        
        print("\n" + "="*70)
        print("✅ DATASET UPLOADED SUCCESSFULLY")
        print("="*70)
        print(f"🌐 View at: https://huggingface.co/datasets/{dataset_repo_id}")
        print(f"📊 Episodes: {dataset.num_episodes}")
        print(f"🎬 Total Frames: {dataset.num_frames}")
        print("="*70)
        
    except Exception as e:
        logger.error(f"❌ Upload failed: {e}")
        print(f"\n⚠️  Troubleshooting:")
        print(f"   1. Make sure you're logged in: huggingface-cli login")
        print(f"   2. Check your internet connection")
        print(f"   3. Verify the repo ID: {dataset_repo_id}")
        traceback.print_exc()


---

## 📚 QUICK REFERENCE

### 실행 순서
```
1️⃣  라이브러리 임포트 (셀 1-3)
    ↓
2️⃣  SO100 연결 (SECTION 1)
    ↓
3️⃣  데이터셋 초기화 (SECTION 1B) ⭐ create 또는 resume 자동선택
    ├─ config.yaml 자동 로드
    ├─ 기존 데이터 없음? → LeRobotDataset.create() 🆕
    ├─ 기존 데이터 있음? → LeRobotDataset.resume() ➕
    └─ image_writer 설정 적용
    ↓
4️⃣  VR 서버 시작 (SECTION 2)
    ├─ HTTPS 웹 UI 서버 (8443)
    ├─ WebSocket 서버 (8442)
    ├─ URL 복사
    └─ 헤드셋 연결
    ↓
5️⃣  데이터 수집 시작 (SECTION 3) ⭐ UPDATED
    ├─ VR 모니터링
    ├─ 로봇 제어 루프 (50Hz)
    ├─ 매 프레임 데이터 저장
    │  ├─ observation 수집
    │  ├─ action 수집
    │  ├─ build_dataset_frame() 처리
    │  └─ dataset.add_frame() 추가
    ├─ 에피소드 단위 저장 (dataset.save_episode())
    └─ 설정 에피소드 수 도달 시 종료
    ↓
6️⃣  데이터셋 업로드 (DATASET UPLOAD) ⭐ NEW
    ├─ Hugging Face Hub 로그인 확인
    ├─ dataset.push_to_hub()
    └─ 온라인 저장소 생성/업데이트
    ↓
7️⃣  정리 (CLEANUP)
```

### 💾 데이터 수집 형식 (record.py와 동일)
```python
# 매 프레임마다:
observation = robot.get_observation()
action = controller.p_control_action(robot)
sent_action = robot.send_action(action)

# 데이터셋 저장
observation_frame = build_dataset_frame(dataset.features, observation, "observation")
action_frame = build_dataset_frame(dataset.features, sent_action, "action")
frame = {**observation_frame, **action_frame, "task": single_task}
dataset.add_frame(frame)

# 에피소드 완료 시
dataset.save_episode()
```

### 🔄 데이터셋 모드 선택 (자동)

**신규 데이터셋 생성:**
```python
dataset = LeRobotDataset.create(
    repo_id="my_dataset",
    fps=30,
    root="./datasets",
    features=dataset_features
)
```

**기존 데이터셋에 추가:**
```python
dataset = LeRobotDataset.resume(
    repo_id="my_dataset",
    root="./datasets"  # 반드시 필요 (root 필수)
)
```

💡 노트북은 자동으로 상황에 맞춰 선택합니다:
- 폴더 없음 → `create()` 🆕
- 폴더 있음 → `resume()` ➕

### 🎮 컨트롤러 맵핑
| VR 입력 | 로봇 제어 |
|--------|----------|
| 컨트롤러 위치 | 팔의 end-effector 위치 |
| 손목 각도 | 손목 회전 (roll/flex) |
| 트리거 (> 50%) | 그리퍼 개폐 |

### 📊 데이터셋 구조
```
dataset/
├── repo_id/
│   ├── episode_0/
│   │   ├── data.parquet      # 모든 프레임 데이터
│   │   └── episode_metadata.json
│   ├── episode_1/
│   ├── episode_2/            # resume()으로 추가된 에피소드
│   └── ...
├── meta/
│   ├── info.json
│   ├── stats.json
│   └── tasks.parquet
└── videos/ (옵션)
```

### 🌐 Hugging Face Hub 워크플로우
```
첫 수집 세션 (SECTION 3)
    ↓
dataset.add_frame() × N
    ↓
dataset.save_episode()
    ↓
dataset.push_to_hub() (DATASET UPLOAD)
    ↓
🌍 https://huggingface.co/datasets/{repo_id}

---

두 번째 수집 세션 (resume 모드)
    ↓
SECTION 1B → LeRobotDataset.resume() ✅
    ↓
dataset.add_frame() × M (새로운 데이터)
    ↓
dataset.save_episode() (새로운 에피소드 추가)
    ↓
dataset.push_to_hub() (기존 데이터 + 신규 에피소드 업로드)
    ↓
🌍 동일 repo에 에피소드만 추가됨
```

### ⚙️ config.yaml 설정 (SECTION 1B가 자동 로드)
```yaml
# 📁 /home/choyunsang/lerobot/config.yaml
dataset:
  repo_id: "your_username/so100_vr_data"      # Hugging Face repo ID
  single_task: "VR teleoperation task"        # 작업 설명
  root: "/home/choyunsang/Dataset"            # 저장 경로
  fps: 30                                      # 프레임율
  num_episodes: 5                             # 총 에피소드 수

image_writer:
  num_processes: 0                            # 프로세스 수
  threads_per_camera: 4                       # 스레드 수
  batch_encoding_size: 1                      # 배치 인코딩 크기

control:
  kp: 0.5                                     # P 제어 게인
```

### 🚨 문제 해결

**"Config file not found" 경고**
- ✅ 파일 경로 확인: `/home/choyunsang/lerobot/config.yaml`
- ✅ 파일이 없으면 기본값 사용 (하드코딩 값)
- ✅ config.yaml을 만들어서 설정 저장

**"Dataset already exists" 메시지 (정상)**
- ✅ 이전 수집 세션의 데이터가 있다는 뜻
- ✅ `resume()` 모드로 자동 전환
- ✅ 기존 에피소드 보존, 새 데이터만 추가

**"No dataset found" 에러 (SECTION 3)**
- ✅ SECTION 1B를 먼저 실행했는지 확인
- ✅ config.yaml에서 값이 제대로 로드되었는지 확인

**"Upload failed" (DATASET UPLOAD)**
- ✅ `huggingface-cli login` 실행
- ✅ repo_id에 정확한 username 입력
- ✅ config.yaml의 repo_id 확인
- ✅ 인터넷 연결 확인

**"Frames not saving"**
- ✅ SECTION 3의 loop_count 증가 확인
- ✅ dataset 객체가 None이 아닌지 확인
- ✅ robot.get_observation() 정상 작동 확인
- ✅ config.yaml의 root 경로에 쓰기 권한 확인

**설정 변경하고 싶어요**
- ✅ `/home/choyunsang/lerobot/config.yaml` 파일 편집
- ✅ 원하는 값 변경
- ✅ SECTION 1B 다시 실행 (설정 재로드)

**여러 번 수집하려면?**
- ✅ 첫 번째: SECTION 1B → `create()` 🆕 → SECTION 3 수집
- ✅ 두 번째: SECTION 1B → `resume()` ➕ → SECTION 3 수집
- ✅ 자동으로 기존 데이터 보존 + 신규 에피소드 추가
